# Arabic Dialect — Hierarchical Preprocessing

Adds a **region_label** column on top of your existing dialect labels.
The 13-step cleaning pipeline is completely unchanged.

**Region tree:**
```
Root (region)
 ├── levantine   → Syria, Lebanon, Palestine, Jordan, Iraq
 ├── maghrebi    → Morocco, Algeria, Tunisia, Libya
 ├── gulf        → Saudi, Kuwait, Emirates, Bahrain, Qatar, Oman
 └── nile_basin  → Egypt, Sudan, Yemen
```

**Output files (per split):**
- `text` — cleaned Arabic text
- `dialect_label` — integer (city-level, 18 classes)
- `region_label`  — integer (region-level, 4 classes)  ← NEW
- `sentiment_label` — integer (0=negative, 1=positive)


## 1. Imports & Setup

In [ ]:
import os, re, json, warnings, unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

try:
    import emoji as emoji_lib; EMOJI_OK = True
except ImportError:
    EMOJI_OK = False; print('[WARN] pip install emoji')

try:
    import ftfy; FTFY_OK = True
except ImportError:
    FTFY_OK = False; print('[WARN] pip install ftfy  (strongly recommended)')

print('[OK] Imports complete.')

[WARN] pip install emoji
[WARN] pip install ftfy  (strongly recommended)
[OK] Imports complete.


## 2. Configuration

In [ ]:
# ── FILE PATH ─────────────────────────────────────────────────────────────────
AD_FILE        = '/content/Multi-dialect Arabic dataset.csv'
AD_TEXT_COL    = 'Comment'
AD_LABEL_COL   = 'label'
AD_DIALECT_COL = 'dialect'

# ── OUTPUT ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/hierarchical_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── PREPROCESSING OPTIONS ─────────────────────────────────────────────────────
DROP_NEUTRAL  = True    # drop neutral sentiment rows
DROP_DUPES    = True    # drop duplicate comments
EMOJI_MODE    = 'remove'  # 'remove' | 'demojize'
MIN_TOKENS    = 2       # drop rows with fewer tokens after cleaning
RANDOM_SEED   = 32
TEST_SIZE     = 0.15
VAL_SIZE      = 0.15

# ── REGION MAPPING ────────────────────────────────────────────────────────────
# This is the only new thing compared to your original preprocessing.
# Maps each dialect (city-level) to its region (root-level).
DIALECT_TO_REGION = {
    # Levantine
    'Syria':     'levantine',
    'Lebanon':   'levantine',
    'Palestine': 'levantine',
    'Jordan':    'levantine',
    'Iraq':      'levantine',
    # Maghrebi
    'Moroccan':  'maghrebi',
    'Algeria':   'maghrebi',
    'Tunisia':   'maghrebi',
    'Libya':     'maghrebi',
    # Gulf  (kept separate — no merging needed in hierarchical setup)
    'Saudi':     'gulf',
    'Kuwait':    'gulf',
    'Emirates':  'gulf',
    'Bahrain':   'gulf',
    'Qaṭar':     'gulf',   # note: has diacritic in your data
    'Oman':      'gulf',
    # Nile Basin
    'Egypt':     'nile_basin',
    'Sudanese':  'nile_basin',
    'Yemen':     'nile_basin',
}

# Region integer encoding (for the root classifier)
REGION_TO_ID = {
    'levantine':  0,
    'maghrebi':   1,
    'gulf':       2,
    'nile_basin': 3,
}

print(f'[CONFIG] Output dir : {OUTPUT_DIR}')
print(f'[CONFIG] Regions    : {list(REGION_TO_ID.keys())}')

[CONFIG] Output dir : /content/hierarchical_outputs
[CONFIG] Regions    : ['levantine', 'maghrebi', 'gulf', 'nile_basin']


## 3. Regex Patterns (unchanged)

In [ ]:
_AR_INDIC     = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
_EXT_AR_INDIC = str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789')

RE_URL       = re.compile(r'https?://\S+|www\.\S+')
RE_MENTION   = re.compile(r'@\w+')
RE_HASHTAG   = re.compile(r'#(\w+)')
RE_FLAG      = re.compile(r'[\U0001F1E6-\U0001F1FF]{2}')
RE_EMOJI_FB  = re.compile(r'[\U00010000-\U0010ffff]', flags=re.UNICODE)
RE_TASHKEEL  = re.compile(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]')
RE_TATWEEL   = re.compile(r'\u0640')
RE_OBFUSC_AR = re.compile(r'(?<=[\u0600-\u06FF])([^\u0600-\u06FF\w\s]+)(?=[\u0600-\u06FF])')
RE_ELONGATE  = re.compile(r'(.)\1{3,}')
RE_REPUNCT   = re.compile(r'([!?.,،؛؟]){2,}')

print('[OK] Regex compiled.')

[OK] Regex compiled.


## 4. Encoding Fix Utilities (unchanged)

In [ ]:
def fix_encoding(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    if FTFY_OK:
        fixed = ftfy.fix_text(text)
        if fixed != text and re.search(r'[\u0600-\u06FF]', fixed):
            return fixed
    for enc in ('latin-1', 'cp1252'):
        try:
            fixed = text.encode(enc).decode('utf-8')
            if re.search(r'[\u0600-\u06FF]', fixed):
                return fixed
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return text

## 5. 13-Step Cleaning Pipeline (unchanged)

In [ ]:
def unicode_normalize(text):
    if not isinstance(text, str): return ''
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', text)).strip()

def convert_arabic_indic_digits(text):
    return text.translate(_AR_INDIC).translate(_EXT_AR_INDIC)

def remove_flag_emojis(text):
    return RE_FLAG.sub(' ', text)

def handle_emojis(text, mode='remove'):
    if mode == 'demojize' and EMOJI_OK:
        return emoji_lib.demojize(text, language='en')
    if EMOJI_OK:
        return emoji_lib.replace_emoji(text, replace=' ')
    return RE_EMOJI_FB.sub(' ', text)

def remove_urls_and_mentions(text):
    return RE_MENTION.sub(' ', RE_URL.sub(' ', text))

def expand_hashtags(text):
    return RE_HASHTAG.sub(r' \1 ', text)

def deobfuscate_arabic(text):
    prev = None
    while prev != text:
        prev = text
        text = RE_OBFUSC_AR.sub('', text)
    return text

def fix_fragmented_arabic(text):
    tokens = text.split(' ')
    result, i = [], 0
    while i < len(tokens):
        tok = tokens[i]
        if len(tok) == 1 and re.match(r'[\u0600-\u06FF]', tok):
            run, j = [tok], i + 1
            while j < len(tokens) and len(tokens[j]) == 1 and re.match(r'[\u0600-\u06FF]', tokens[j]):
                run.append(tokens[j]); j += 1
            result.append(''.join(run) if len(run) >= 2 else tok)
            i = j if len(run) >= 2 else i + 1
        else:
            result.append(tok); i += 1
    return ' '.join(result)

def remove_tashkeel(text):
    return RE_TATWEEL.sub('', RE_TASHKEEL.sub('', text))

def normalize_arabic_letters(text):
    text = re.sub(r'[إأٱآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    return text

def collapse_elongation(text):
    return RE_ELONGATE.sub(r'\1\1\1', text)

def normalize_punctuation(text):
    return RE_REPUNCT.sub(r'\1', text)

def final_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text, emoji_mode='remove'):
    """Full 13-step Arabic cleaning pipeline."""
    text = fix_encoding(text)
    text = unicode_normalize(text)           # step 1
    text = convert_arabic_indic_digits(text) # step 2
    text = remove_flag_emojis(text)          # step 3
    text = handle_emojis(text, emoji_mode)   # step 4
    text = remove_urls_and_mentions(text)    # step 5
    text = expand_hashtags(text)             # step 6
    text = deobfuscate_arabic(text)          # step 7
    text = fix_fragmented_arabic(text)       # step 8
    text = remove_tashkeel(text)             # step 9
    text = normalize_arabic_letters(text)    # step 10
    text = collapse_elongation(text)         # step 11
    text = normalize_punctuation(text)       # step 12
    text = final_whitespace(text)            # step 13
    return text

print('[OK] 13-step cleaning pipeline ready.')

[OK] 13-step cleaning pipeline ready.


## 6. Load & Clean Data

In [ ]:
print(f'[LOAD] Reading: {AD_FILE}')
df = pd.read_csv(AD_FILE)
print(f'[LOAD] Raw shape: {df.shape}')

# Standardize column names
df.columns = [c.lower().strip() for c in df.columns]
df = df.rename(columns={'comment': 'text', AD_LABEL_COL: 'sentiment_raw',
                         AD_DIALECT_COL: 'dialect'})

# Drop rows with missing core fields
df = df.dropna(subset=['text', 'dialect', 'sentiment_raw']).reset_index(drop=True)

# ── Sentiment encoding ─────────────────────────────────────────────

# Normalize labels
df['sentiment_raw'] = (
    df['sentiment_raw']
    .astype(str)
    .str.strip()
    .str.upper()
)

# Encode labels
label_mapping = {
    'P': 1,
    'N': 0
}

df['sentiment_label'] = df['sentiment_raw'].map(label_mapping)

# Remove unknown labels only
before = len(df)
df = df.dropna(subset=['sentiment_label']).reset_index(drop=True)

print(f"[FILTER] Dropped {before - len(df)} unknown labels.")
print(df['sentiment_label'].value_counts())

# ── Clean text ────────────────────────────────────────────────────────────────
print('[CLEAN] Applying 13-step pipeline...')
df['text'] = df['text'].astype(str).apply(lambda t: clean_text(t, EMOJI_MODE))

# Drop short texts
before = len(df)
df = df[df['text'].str.split().str.len() >= MIN_TOKENS].reset_index(drop=True)
print(f'[FILTER] Dropped {before - len(df)} short texts (< {MIN_TOKENS} tokens).')

# Drop duplicates
if DROP_DUPES:
    before = len(df)
    df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
    print(f'[FILTER] Dropped {before - len(df)} duplicate comments.')

print(f'[CLEAN] Done. Final shape: {df.shape}')
print(df[['text','dialect','sentiment_label']].head(3))

[LOAD] Reading: /content/Multi-dialect Arabic dataset.csv
[LOAD] Raw shape: (7624, 5)
[FILTER] Dropped 370 unknown labels.
sentiment_label
1.0    3629
0.0    3625
Name: count, dtype: int64
[CLEAN] Applying 13-step pipeline...
[FILTER] Dropped 64 short texts (< 2 tokens).
[FILTER] Dropped 15 duplicate comments.
[CLEAN] Done. Final shape: (7175, 6)
                                           text dialect  sentiment_label
0  خطية صغيرة بعدها وهذا اخوها علي اساس غسل عار    Iraq              1.0
1             ليش هيج تتصرف وياها هاي خلقه ربنا    Iraq              1.0
2           بالعكس البنيه تجنن وسمارها كولش حلو    Iraq              1.0


In [ ]:
df

,text,dialect,platform,category,sentiment_raw,sentiment_label
0,خطية صغيرة بعدها وهذا اخوها علي اساس غسل عار,Iraq,YT,Feminist,P,1.0
1,ليش هيج تتصرف وياها هاي خلقه ربنا,Iraq,YT,Feminist,P,1.0
2,بالعكس البنيه تجنن وسمارها كولش حلو,Iraq,YT,Feminist,P,1.0
3,والله فدوه لحلكج يا امي انتي هاي المراه العراق...,Iraq,YT,Feminist,P,1.0
4,يمه فدوه للنسوان المعدله,Iraq,YT,Feminist,P,1.0
...,...,...,...,...,...,...
7170,ردونا احنا الظالمين ارض الله واسعه واحنا مش نا...,Tunisia,FB,Political,N,0.0
7171,تحيا تونس ويحيا قيس سعيد ويسقط الخونة,Tunisia,FB,Political,N,0.0
7172,ودائما يبقي العديد ضءيل بالنسبة الاعدادهم المه...,Tunisia,FB,Political,N,0.0
7173,لا نريد عودة طوعية اللي ماعندوش وراق اروح وبال...,Tunisia,FB,Political,N,0.0


## 7. Add Region Labels  ← THE NEW STEP

In [ ]:
# Map dialect → region name
df['region'] = df['dialect'].map(DIALECT_TO_REGION)

# Check for any unmapped dialects
unmapped = df[df['region'].isna()]['dialect'].unique()
if len(unmapped) > 0:
    print(f'[WARN] Unmapped dialects (will be dropped): {unmapped}')
    df = df.dropna(subset=['region']).reset_index(drop=True)
else:
    print('[OK] All dialects mapped to regions successfully.')

# Encode dialect → integer
dialect_names = sorted(df['dialect'].unique())
DIALECT_TO_ID = {d: i for i, d in enumerate(dialect_names)}
ID_TO_DIALECT = {i: d for d, i in DIALECT_TO_ID.items()}
df['dialect_label'] = df['dialect'].map(DIALECT_TO_ID).astype(int)

# Encode region → integer
df['region_label'] = df['region'].map(REGION_TO_ID).astype(int)
ID_TO_REGION = {v: k for k, v in REGION_TO_ID.items()}

print(f'\n[LABELS] Dialect classes : {len(DIALECT_TO_ID)}')
print(f'[LABELS] Region  classes : {len(REGION_TO_ID)}')
print(f'\nDialect → Region mapping:')
for d, r in sorted(DIALECT_TO_REGION.items()):
    if d in DIALECT_TO_ID:
        print(f'  {d:<12} → {r}  (id={DIALECT_TO_ID[d]})')

print(f'\nRegion distribution:')
print(df['region'].value_counts())
print(f'\nDialect distribution:')
print(df['dialect'].value_counts())

[OK] All dialects mapped to regions successfully.

[LABELS] Dialect classes : 18
[LABELS] Region  classes : 4

Dialect → Region mapping:
  Algeria      → maghrebi  (id=0)
  Bahrain      → gulf  (id=1)
  Egypt        → nile_basin  (id=2)
  Emirates     → gulf  (id=3)
  Iraq         → levantine  (id=4)
  Jordan       → levantine  (id=5)
  Kuwait       → gulf  (id=6)
  Lebanon      → levantine  (id=7)
  Libya        → maghrebi  (id=8)
  Moroccan     → maghrebi  (id=9)
  Oman         → gulf  (id=10)
  Palestine    → levantine  (id=11)
  Qaṭar        → gulf  (id=12)
  Saudi        → gulf  (id=13)
  Sudanese     → nile_basin  (id=14)
  Syria        → levantine  (id=15)
  Tunisia      → maghrebi  (id=16)
  Yemen        → nile_basin  (id=17)

Region distribution:
region
levantine     2601
maghrebi      2091
nile_basin    1572
gulf           911
Name: count, dtype: int64

Dialect distribution:
dialect
Sudanese     528
Libya        527
Saudi        525
Jordan       525
Moroccan     523
Syria    

## 8. Train / Val / Test Split

In [ ]:
# Stratify on dialect_label to keep distribution balanced across splits
train_df, temp_df = train_test_split(
    df, test_size=0.3,
    stratify=df['dialect_label'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    stratify=temp_df['dialect_label'], random_state=RANDOM_SEED
)

# Final columns
COLS = ['text', 'dialect_label', 'region_label', 'sentiment_label']
train_df = train_df[COLS].reset_index(drop=True)
val_df   = val_df[COLS].reset_index(drop=True)
test_df  = test_df[COLS].reset_index(drop=True)

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')
print(f'\nTrain region distribution:')
print(train_df['region_label'].value_counts().rename(ID_TO_REGION))
print(f'\nTrain dialect distribution:')
print(train_df['dialect_label'].value_counts().rename(ID_TO_DIALECT))

Train : 5,022
Val   : 1,076
Test  : 1,077

Train region distribution:
region_label
levantine     1821
maghrebi      1463
nile_basin    1100
gulf           638
Name: count, dtype: int64

Train dialect distribution:
dialect_label
Sudanese     370
Libya        369
Jordan       367
Saudi        367
Moroccan     366
Syria        366
Yemen        365
Algeria      365
Lebanon      365
Egypt        365
Tunisia      363
Iraq         362
Palestine    361
Kuwait        92
Emirates      92
Bahrain       35
Qaṭar         27
Oman          25
Name: count, dtype: int64


## 9. Save CSVs & Label Maps

In [ ]:
train_df.to_csv(f'{OUTPUT_DIR}/train.csv', index=False, encoding='utf-8-sig')
val_df.to_csv(f'{OUTPUT_DIR}/val.csv',   index=False, encoding='utf-8-sig')
test_df.to_csv(f'{OUTPUT_DIR}/test.csv', index=False, encoding='utf-8-sig')

# Save label maps
with open(f'{OUTPUT_DIR}/dialect_map.json', 'w', encoding='utf-8') as f:
    json.dump({'dialect_to_id': DIALECT_TO_ID, 'id_to_dialect': ID_TO_DIALECT}, f,
              ensure_ascii=False, indent=2)

with open(f'{OUTPUT_DIR}/region_map.json', 'w', encoding='utf-8') as f:
    json.dump({'region_to_id': REGION_TO_ID, 'id_to_region': ID_TO_REGION,
               'dialect_to_region': DIALECT_TO_REGION}, f,
              ensure_ascii=False, indent=2)

print(f'[SAVE] train.csv  → {OUTPUT_DIR}/train.csv')
print(f'[SAVE] val.csv    → {OUTPUT_DIR}/val.csv')
print(f'[SAVE] test.csv   → {OUTPUT_DIR}/test.csv')
print(f'[SAVE] dialect_map.json')
print(f'[SAVE] region_map.json')
print(f'\n[DONE] Preprocessing complete.')
print(f'       Your data now has both dialect_label (18 classes) and region_label (4 classes).')

[SAVE] train.csv  → /content/hierarchical_outputs/train.csv
[SAVE] val.csv    → /content/hierarchical_outputs/val.csv
[SAVE] test.csv   → /content/hierarchical_outputs/test.csv
[SAVE] dialect_map.json
[SAVE] region_map.json

[DONE] Preprocessing complete.
       Your data now has both dialect_label (18 classes) and region_label (4 classes).


## 10. Quick Sanity Check

In [ ]:
print('=== Sample rows from train split ===')
sample = train_df.sample(5, random_state=42)
for _, row in sample.iterrows():
    print(f'  text     : {row["text"][:60]}...')
    print(f'  dialect  : {ID_TO_DIALECT[row["dialect_label"]]} (id={row["dialect_label"]})')
    print(f'  region   : {ID_TO_REGION[row["region_label"]]} (id={row["region_label"]})')
    print(f'  sentiment: {row["sentiment_label"]}')
    print()

=== Sample rows from train split ===
  text     : تفو عليك يا خانزة لكان جيتي مراة تعاوني والديك مشي تحشمي بيه...
  dialect  : Algeria (id=0)
  region   : maghrebi (id=1)
  sentiment: 0.0

  text     : سلعت او زبال ديال صحاب لهريسا والطبونية ممنوع دخول لبلادنا م...
  dialect  : Moroccan (id=9)
  region   : maghrebi (id=1)
  sentiment: 0.0

  text     : زلم تناتيح ما شاء الله...
  dialect  : Libya (id=8)
  region   : maghrebi (id=1)
  sentiment: 1.0

  text     : بعد عمري والله ❤️...
  dialect  : Emirates (id=3)
  region   : gulf (id=2)
  sentiment: 1.0

  text     : ختك نجيك لتونس نيكك...
  dialect  : Tunisia (id=16)
  region   : maghrebi (id=1)
  sentiment: 0.0

